# Final Comparison: Decision Tree, AdaBoost, and Random Forest

**Person 4 — project synthesis**

This notebook consolidates the team's implementation and experiment results into one reproducible comparison. It answers the central question: **under what conditions does boosting outperform bagging, and vice versa, and why?**

The analysis uses the project's saved result files rather than hard-coded scores. Re-run `python src/experiments/run_all.py --include-optional --continue-on-error` from the repository root before executing this notebook to refresh every available result.

## 1. Scope and evidence

The synthesis draws on:

- Person 1's Decision Tree notebook (`DecisionTreeNotebook.ipynb`): CART behavior, stump underfitting, feature importance, and sklearn parity.
- AdaBoost scaling outputs in `results/adaboost_scaling.csv`.
- Person 3's Random Forest notebook (`random_forest_experiments.ipynb`): estimator/depth scaling, noise robustness, OOB behavior, and parallel execution.
- Canonical result tables produced by the experiment modules.
- The optional exploration notebook when it contains interactive content.

> Some member notebook filenames differ from the planned names, and the repository currently has no separate Person 2 notebook. The integrated AdaBoost result files are therefore the authoritative evidence for that model.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'results').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the project repository.')

ROOT = find_repo_root(Path.cwd().resolve())
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (9, 5), 'axes.titleweight': 'bold', 'figure.dpi': 110})
COLORS = {'Decision Tree': '#4C78A8', 'Stump': '#9ecae9', 'AdaBoost': '#F58518', 'Random Forest': '#54A24B', 'sklearn': '#B279A2'}

def read_csv(name: str) -> pd.DataFrame:
    path = RESULTS / name
    if not path.exists():
        warnings.warn(f'Missing {path.relative_to(ROOT)}; run the corresponding experiment.')
        return pd.DataFrame()
    return pd.read_csv(path)

baseline = read_csv('baseline_results.csv')
ada_scaling = read_csv('adaboost_scaling.csv')
rf_scaling = read_csv('rf_scaling_results.csv')
noise = read_csv('noise_robustness_results.csv')
head_to_head = read_csv('head_to_head_results.csv')
bias_variance = read_csv('bias_variance_results.csv')
parallel = read_csv('rf_parallel_benchmark.csv')

print('Repository:', ROOT)
print('Loaded rows:', {'baseline': len(baseline), 'AdaBoost scaling': len(ada_scaling), 'RF scaling': len(rf_scaling), 'noise': len(noise)})

## 2. Baseline: one tree versus one stump

A full tree is flexible enough to model nonlinear interactions, while a depth-1 stump has deliberately high bias. The stump is not intended to win alone: it is the atomic learner that AdaBoost improves sequentially. Person 1's notebook also reports close agreement with sklearn, supporting the correctness of the from-scratch CART implementation.

In [ ]:
if not baseline.empty:
    name_map = {'MyTree': 'Decision Tree', 'MyStump': 'Stump', 'sklearn': 'sklearn'}
    base = baseline.assign(model_label=baseline['model'].map(name_map).fillna(baseline['model']))
    metric_cols = [c for c in ['accuracy', 'macro_f1', 'roc_auc'] if c in base]
    display(base[['dataset', 'model_label', *metric_cols]].round(4).rename(columns={'model_label': 'model'}))

    plot_df = base.groupby('model_label')[metric_cols].mean().reindex(['Stump', 'Decision Tree', 'sklearn']).dropna(how='all')
    ax = plot_df.plot(kind='bar', color=['#4C78A8', '#F58518', '#54A24B'], width=0.75)
    ax.set(title='Baseline mean performance across available datasets', xlabel='', ylabel='Score', ylim=(0, 1.05))
    ax.tick_params(axis='x', rotation=0)
    ax.legend(title='Metric', loc='lower right')
    plt.tight_layout(); plt.show()
else:
    display(Markdown('**Baseline results unavailable.**'))

### Interpretation

- **Decision Tree:** low training bias and strong interpretability, but split decisions are greedy and a deep tree can change substantially after a small data perturbation. This instability is high variance.
- **Decision Stump:** stable and inexpensive, but one boundary cannot represent complex class structure. Its high bias creates room for boosting.
- **Validation:** similarity between the custom tree and sklearn baseline is a sanity check, not use of sklearn as the primary implementation.

## 3. AdaBoost scaling: reducing the bias of weak learners

AdaBoost trains stumps sequentially. After each round it increases the weight of misclassified observations, forcing the next stump to focus on examples the current ensemble finds difficult. The final classifier is a weighted vote, with more reliable stumps receiving larger weights. This additive process turns many simple boundaries into a flexible decision function.

In [ ]:
if not ada_scaling.empty:
    datasets = list(ada_scaling['dataset'].drop_duplicates())
    fig, axes = plt.subplots(1, len(datasets), figsize=(5 * len(datasets), 4), squeeze=False, sharey=True)
    for ax, dataset in zip(axes.ravel(), datasets):
        part = ada_scaling[ada_scaling['dataset'] == dataset].sort_values('n_estimators')
        ax.plot(part['n_estimators'], part['train_accuracy'], label='Train', lw=2)
        ax.plot(part['n_estimators'], part['test_accuracy'], label='Test', lw=2)
        best = part.loc[part['test_accuracy'].idxmax()]
        ax.scatter(best['n_estimators'], best['test_accuracy'], color='black', zorder=3)
        ax.set(title=f'{dataset}: best test={best.test_accuracy:.3f} @ {int(best.n_estimators)}', xlabel='Estimators', ylabel='Accuracy', ylim=(0, 1.03))
    axes.ravel()[0].legend()
    fig.suptitle('AdaBoost learning curves', y=1.04, fontweight='bold')
    plt.tight_layout(); plt.show()

    ada_best = (ada_scaling.loc[ada_scaling.groupby('dataset')['test_accuracy'].idxmax(), ['dataset', 'n_estimators', 'test_accuracy', 'test_macro_f1']]
                .sort_values('dataset').reset_index(drop=True))
    display(Markdown('**Best observed AdaBoost checkpoint per dataset**'))
    display(ada_best.round(4))

### What the curves mean

Training performance should improve as rounds are added because model capacity increases. Test performance commonly improves quickly and then plateaus or oscillates. A widening train–test gap signals that later rounds are specializing in hard points or label noise. AdaBoost is therefore strongest when weak learners are slightly better than chance and labels are reasonably clean; it is less forgiving when repeatedly up-weighted observations are mislabeled or extreme outliers.

## 4. Random Forest scaling: variance reduction through diversity

Random Forest trains full trees independently on bootstrap samples and selects a random subset of features at every split. Bootstrapping changes the data seen by each tree; feature subsampling reduces correlation between trees. Majority voting then averages away unstable errors while retaining the expressive power of deep trees.

In [ ]:
if not rf_scaling.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    est = rf_scaling[rf_scaling['sweep'] == 'n_estimators'].copy()
    for dataset, part in est.groupby('dataset'):
        part = part.sort_values('n_estimators')
        axes[0].plot(part['n_estimators'], part['test_accuracy'], marker='o', ms=3, label=dataset)
    axes[0].set(title='Forest size: diminishing returns', xlabel='Number of trees', ylabel='Test accuracy', ylim=(0, 1.03))
    axes[0].legend(fontsize=8)

    depth = rf_scaling[rf_scaling['sweep'] == 'max_depth'].copy()
    for dataset, part in depth.groupby('dataset'):
        part = part.sort_values('max_depth')
        axes[1].plot(part['max_depth'], part['test_accuracy'], marker='o', ms=3, label=dataset)
    axes[1].set(title='Tree depth: bias–variance control', xlabel='Maximum depth', ylabel='Test accuracy', ylim=(0, 1.03))
    axes[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()

    if 'oob_accuracy' in est and est['oob_accuracy'].notna().any():
        oob = est.dropna(subset=['oob_accuracy'])
        display(Markdown(f"Across recorded forest-size runs, mean absolute OOB–test accuracy gap: **{(oob['oob_accuracy'] - oob['test_accuracy']).abs().mean():.3f}**."))

### Scalability trade-offs

- More trees usually stabilize accuracy and OOB estimates, but improvements diminish while fit and prediction costs continue to grow.
- Increasing depth first reduces bias; beyond the useful range it mainly increases per-tree variance and computation. Averaging makes a forest less sensitive to deep-tree overfitting than a single tree.
- Trees are independent, so Random Forest fitting can be parallelized. AdaBoost rounds are sequential because each round depends on the previous sample weights.
- OOB accuracy provides a convenient internal generalization estimate without a separate validation split, although it does not replace final held-out or cross-validated evaluation.

## 5. Robustness to label noise

The controlled experiment corrupts training labels but evaluates against a clean test set. This isolates sensitivity to supervision noise.

In [ ]:
if not noise.empty:
    valid = noise[noise.get('status', 'ok').eq('ok')].copy() if 'status' in noise else noise.copy()
    model_names = {'team_random_forest': 'Random Forest', 'team_adaboost': 'AdaBoost'}
    valid['model_label'] = valid['model'].map(model_names).fillna(valid['model'])
    datasets = list(valid['dataset'].drop_duplicates())
    ncols = min(3, len(datasets)); nrows = int(np.ceil(len(datasets) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.8 * nrows), squeeze=False, sharey=True)
    for ax, dataset in zip(axes.ravel(), datasets):
        for model, part in valid[valid['dataset'] == dataset].groupby('model_label'):
            part = part.sort_values('noise_fraction')
            ax.plot(part['noise_fraction'] * 100, part['accuracy'], marker='o', lw=2, label=model, color=COLORS.get(model))
        ax.set(title=dataset.replace('_', ' '), xlabel='Flipped training labels (%)', ylabel='Clean-test accuracy', ylim=(0, 1.03))
        ax.legend(fontsize=8)
    for ax in axes.ravel()[len(datasets):]: ax.axis('off')
    fig.suptitle('Noise robustness: boosting versus bagging', y=1.01, fontweight='bold')
    plt.tight_layout(); plt.show()

    clean = valid[valid['noise_fraction'] == 0][['dataset', 'model_label', 'accuracy']].rename(columns={'accuracy': 'clean_accuracy'})
    worst = valid.loc[valid.groupby(['dataset', 'model_label'])['noise_fraction'].idxmax(), ['dataset', 'model_label', 'noise_fraction', 'accuracy']]
    degradation = worst.merge(clean, on=['dataset', 'model_label'], how='left')
    degradation['accuracy_change'] = degradation['accuracy'] - degradation['clean_accuracy']
    display(degradation.sort_values(['dataset', 'model_label']).round(4))

### Why the models respond differently

AdaBoost deliberately concentrates weight on persistent mistakes. On clean data this targets difficult decision regions; under label noise it can spend later rounds chasing examples that cannot be fitted consistently. Random Forest does not adaptively amplify individual mistakes. A corrupted point appears in only some bootstrap samples, and averaging dilutes its influence, so bagging is generally expected to degrade more gracefully. The plotted dataset-level curves are the evidence for whether that expectation holds in this run—class imbalance and sample size can create exceptions.

## 6. Head-to-head evidence and compact scorecard

The preferred final comparison is the project's fixed-resource 5-fold cross-validation output. If that CSV is not present, the notebook reports a clearly labeled fallback using available clean-test and scaling results; it never invents missing folds or standard deviations.

In [ ]:
if not head_to_head.empty:
    display(head_to_head.round(4))
    numeric = [c for c in ['accuracy', 'macro_f1', 'roc_auc'] if c in head_to_head]
    model_col = 'model' if 'model' in head_to_head else 'model_name'
    summary = head_to_head.groupby(model_col)[numeric].agg(['mean', 'std']).round(4)
    display(summary)
else:
    rows = []
    if not baseline.empty:
        for model, part in baseline.groupby('model'):
            label = {'MyTree': 'Decision Tree', 'MyStump': 'Stump', 'sklearn': 'sklearn Tree'}.get(model, model)
            rows.append({'model': label, 'evidence': 'single split baseline', 'mean_accuracy': part['accuracy'].mean(), 'mean_macro_f1': part['macro_f1'].mean()})
    if not ada_scaling.empty:
        best = ada_scaling.loc[ada_scaling.groupby('dataset')['test_accuracy'].idxmax()]
        rows.append({'model': 'AdaBoost', 'evidence': 'best scaling checkpoint', 'mean_accuracy': best['test_accuracy'].mean(), 'mean_macro_f1': best['test_macro_f1'].mean()})
    if not noise.empty:
        clean_rf = noise[(noise['model'] == 'team_random_forest') & (noise['noise_fraction'] == 0)]
        if not clean_rf.empty:
            rows.append({'model': 'Random Forest', 'evidence': 'clean rows of noise study', 'mean_accuracy': clean_rf['accuracy'].mean(), 'mean_macro_f1': clean_rf['f1_macro'].mean()})
    fallback = pd.DataFrame(rows).sort_values('mean_accuracy', ascending=False)
    display(Markdown('**Fallback descriptive scorecard — protocols differ, so do not treat this as a controlled ranking.**'))
    display(fallback.round(4))

## 7. Bias–variance synthesis

| Model | Typical bias | Typical variance | Noise sensitivity | Parallelism | Main strength | Main limitation |
|---|---:|---:|---:|---:|---|---|
| Decision Stump | High | Low | Low alone | N/A | Fast, transparent weak learner | Severe underfitting |
| Decision Tree | Low when deep | High | Moderate–high | Limited | Interpretable nonlinear rules | Unstable; greedy splits |
| AdaBoost | Reduced across rounds | Can rise on noisy data | High when labels/outliers are bad | Low | Converts weak rules into a strong boundary | Sequential and sensitive to persistent errors |
| Random Forest | Moderate–low | Low after averaging | Usually lower | High | Stable default with OOB estimation | Larger, less interpretable ensemble |

Boosting and bagging solve different weaknesses. Boosting primarily attacks **bias** by correcting previous mistakes; bagging primarily attacks **variance** by averaging diverse, unstable learners. Neither dominates universally.

In [ ]:
if not bias_variance.empty:
    display(bias_variance.round(5))
    required = {'model', 'bias_squared', 'variance'}
    if required.issubset(bias_variance.columns):
        chart = bias_variance.groupby('model')[['bias_squared', 'variance']].mean()
        chart.plot(kind='bar', color=['#F58518', '#54A24B'])
        plt.title('Empirical bias–variance decomposition')
        plt.ylabel('Component magnitude'); plt.xlabel(''); plt.xticks(rotation=20)
        plt.tight_layout(); plt.show()
else:
    display(Markdown('*The run summary records a successful bias–variance experiment, but no canonical `bias_variance_results.csv` is currently available. Add that output to make the empirical decomposition visible here.*'))

## 8. Model selection guide

### Choose a Decision Tree when

- interpretability, simple deployment, or rule inspection is the priority;
- the dataset is small and a constrained depth can be validated;
- the model is serving as a transparent baseline.

### Choose AdaBoost when

- labels are reasonably clean and the stump is consistently better than chance;
- reducing underfitting is more important than parallel training;
- a compact weighted ensemble is desired and estimator count is tuned on validation data.

### Choose Random Forest when

- a robust general-purpose model is needed with limited tuning;
- noisy labels, correlated features, or high single-tree variance are concerns;
- parallel training and OOB validation are useful.

## 9. Final conclusions

1. **The single tree is the interpretability baseline, not the most reliable final predictor.** It can fit complex boundaries but pays for that flexibility with instability.
2. **AdaBoost demonstrates how sequential attention reduces weak-learner bias.** Its largest gains occur early; later estimators should be justified by validation curves, especially when labels are noisy.
3. **Random Forest is the safest default across heterogeneous conditions.** Bootstrap and feature randomness produce diverse trees, and averaging improves generalization while limiting the effect of individual noisy samples.
4. **The winner is condition-dependent.** On clean, learnable boundaries AdaBoost can equal or exceed bagging with stumps. Under label corruption or unstable feature interactions, Random Forest is expected to retain performance better. For explanation-first work, a pruned Decision Tree remains preferable despite lower ensemble-level accuracy.
5. **Evidence must remain protocol-aware.** Final ranking should use the fixed-resource 5-fold head-to-head table; scaling maxima and single-split results explain behavior but are not substitutes for controlled cross-validation.

**Central answer:** boosting wins when the dominant problem is bias and difficult examples are informative; bagging wins when the dominant problem is variance and individual observations or trees are unreliable.

## 10. Reproducibility and project status

The cell below records the latest master-run status alongside the notebook. It is intentionally read-only. The optional `exploration.ipynb` can supply interactive sliders and live plots once populated; the current final comparison uses static figures so it remains portable to HTML/PDF and GitHub.

In [ ]:
summary_path = RESULTS / 'run_all_summary.txt'
if summary_path.exists():
    display(Markdown('```text\n' + summary_path.read_text(encoding='utf-8').strip() + '\n```'))
else:
    display(Markdown('No run summary found. Execute `python src/experiments/run_all.py --include-optional --continue-on-error`.'))

exploration = ROOT / 'notebooks' / 'exploration.ipynb'
print(f'Exploration notebook: {exploration.stat().st_size if exploration.exists() else 0:,} bytes')